In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # input: 3 x 96 x 96
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32 x 48 x 48

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 64 x 24 x 24

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 128 x 12 x 12

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 256 x 6 x 6
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [13]:

img = Image.open("image.png").convert('RGB') # Ensure 3 channels
img.show()



In [15]:
from PIL import Image
from torchvision import transforms

model = SimpleCNN(num_classes=386).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

checkpoint = torch.load('best_model.pth', weights_only=True, map_location=device)
model.load_state_dict(checkpoint)
model.eval()

# 1. Define the transformation pipeline
# Must match what was used during model training
transform = transforms.Compose([
    transforms.Resize((96, 96)),      # Resize to your model's input size
    transforms.ToTensor(),           # Convert to [0, 1] range and tensor format
    transforms.Normalize((0.5,), (0.5,)) # Optional: match your training normalization
])

# 3. Apply transformations and add batch dimension
img_tensor = transform(img)
img_tensor = img_tensor.unsqueeze(0) # Becomes [1, 3, 96, 96]

# 4. Run through the model (from previous step)
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    
# Get the predicted class index
prediction = torch.argmax(output, dim=1).item()
print(f"Predicted Class Index: {prediction}")




Predicted Class Index: 83
